In [3]:
import os
import random
import math
import shutil

def compute_velocity(KE_eV, mass_amu):
    eV_to_J = 1.60218e-19
    amu_to_kg = 1.66054e-27
    KE_J = KE_eV * eV_to_J
    mass_kg = mass_amu * amu_to_kg
    v_m_per_s = math.sqrt(2 * KE_J / mass_kg)
    v_ang_fs = v_m_per_s * 1e-5  # Convert m/s to Å/fs
    return v_ang_fs

def spherical_to_cartesian(r, theta_deg, phi_deg):
    theta = math.radians(theta_deg)
    phi = math.radians(phi_deg)
    x = r * math.sin(theta) * math.cos(phi)
    y = r * math.sin(theta) * math.sin(phi)
    z = r * math.cos(theta)
    return x, y, z

def read_lammps_xyz(filename):
    with open(filename, 'r') as f:
        lines = f.readlines()

    snapshots = []
    i = 0
    while i < len(lines):
        if lines[i].startswith("ITEM: TIMESTEP"):
            i += 1
            i += 1  # "ITEM: NUMBER OF ATOMS"
            assert lines[i].startswith("ITEM: NUMBER OF ATOMS")
            i += 1
            n_atoms = int(lines[i].strip())

            i += 1  # "ITEM: BOX BOUNDS"
            assert lines[i].startswith("ITEM: BOX BOUNDS")
            box_bounds = []
            for _ in range(3):
                i += 1
                box_bounds.append(list(map(float, lines[i].strip().split())))

            i += 1  # "ITEM: ATOMS"
            assert lines[i].startswith("ITEM: ATOMS")
            i += 1

            atoms = []
            for _ in range(n_atoms):
                parts = lines[i].strip().split()
                atom_id = int(parts[0])
                atom_type = 1
                x, y, z = map(float, parts[2:5])
                vx, vy, vz = map(float, parts[5:8])
                atoms.append((atom_id, atom_type, x, y, z, vx, vy, vz))
                i += 1

            snapshots.append({'bounds': box_bounds, 'atoms': atoms})
        else:
            i += 1
    return snapshots

def write_lammps_data(filename, snapshot):
    with open(filename, 'w') as f:
        f.write("# Pd\n")
        f.write("# LAMMPS Description\n\n")
        f.write(f"{len(snapshot['atoms'])} atoms\n")
        f.write("2 atom types\n\n")

        xlo, xhi = snapshot['bounds'][0]
        ylo, yhi = snapshot['bounds'][1]
        zlo, zhi = snapshot['bounds'][2]
        f.write(f"{xlo} {xhi} xlo xhi\n")
        f.write(f"{ylo} {yhi} ylo yhi\n")
        f.write(f"{zlo} {zhi} zlo zhi\n\n")

        f.write("Masses\n\n")
        f.write("1 106.420\n")
        f.write("2 14.001\n\n")

        f.write("Atoms\n\n")
        for atom in snapshot['atoms']:
            # Ensure all atoms have full 8-tuple (with velocities)
            if len(atom) == 5:
                atom = (*atom, 0.0, 0.0, 0.0)
            atom_id, atom_type, x, y, z, vx, vy, vz = atom
            f.write(f"{atom_id} {atom_type} {x:.12f} {y:.12f} {z:.12f}\n")

        f.write("\nVelocities\n\n")
        for atom in snapshot['atoms']:
            if len(atom) == 5:
                atom = (*atom, 0.0, 0.0, 0.0)
            atom_id, atom_type, x, y, z, vx, vy, vz = atom
            f.write(f"{atom_id} {vx:.12f} {vy:.12f} {vz:.12f}\n")

def main():
    input_file = '7.xyz'
    base_dir = 'nvt_inputs'
    os.makedirs(base_dir, exist_ok=True)

    snapshots = read_lammps_xyz(input_file)
    print(f"Total snapshots read: {len(snapshots)}")

    if len(snapshots) < 100:
        raise ValueError("Not enough snapshots to choose 100 unique frames.")

    KE_N_eV = 8.0
    N_mass = 14.001
    N_velocity = compute_velocity(KE_N_eV, N_mass)

    chosen_snapshots = random.sample(snapshots, 100)
    for idx, snap in enumerate(chosen_snapshots, start=1):
        subdir = os.path.join(base_dir, str(idx))
        os.makedirs(subdir, exist_ok=True)

        # Randomly place N2 center of mass in xy-plane
        grid_x = random.uniform(0.0, 8.3644)
        grid_y = random.uniform(0.0, 8.3644)
        COM_z = snap['atoms'][44][4] + 5.0

        # Random orientation
        theta = random.uniform(0, 180)
        phi = random.uniform(0, 360)

        bond_length = 1.10
        half_bond = bond_length / 2
        COM_z_offset = 2.0  # height above surface

        bx, by, bz = spherical_to_cartesian(half_bond, theta, phi)

        center_z = COM_z + COM_z_offset

        # First N atom at +0.55 Å
        dx1, dy1, dz1 = bx, by, bz
        # Second N atom at -0.55 Å
        dx2, dy2, dz2 = -bx, -by, -bz

        atom_id_offset = max(atom[0] for atom in snap['atoms'])
        n1 = (atom_id_offset + 1, 2, grid_x + dx1, grid_y + dy1, COM_z + dz1, 0.0, 0.0, -N_velocity)
        n2 = (atom_id_offset + 2, 2, grid_x + dx2, grid_y + dy2, COM_z + dz2, 0.0, 0.0, -N_velocity)

        snap['atoms'].extend([n1, n2])

        output_filename = os.path.join(subdir, "input.data")
        write_lammps_data(output_filename, snap)
        shutil.copy("vasp.job", os.path.join(subdir, "vasp.job"))

    print(f"✅ 100 folders created with input.data and vasp.job files inside '{base_dir}/[1-100]'")

if __name__ == "__main__":
    main()

Total snapshots read: 101
✅ 100 folders created with input.data and vasp.job files inside 'nvt_inputs/[1-100]'
